## Pupil Gluing Characterisation 🖌️ WITH flight model!

This notebook:
1. Calculates the Zernike coefficients on two separate plates (one with LC toliman pupil)
2. Calculates the Zernike coefficients on the glued stack

Existing system aberrations are loaded and can be calculated using System_Aberrations.ipynb.

________________________________________________________________________________________________
**NOTE (on data orientation)**: When modelling dLux layers (aberrated apertures, transmissive layers, etc) - they are orientated w.r.t wavefront POV in direction of propagation. In our setup, the data collected has two orientation flips:
1. BFS-U3-200S6M flips image upside down from propagation direction POV (and stores it like this too)
2. The pupil plane is focused by a second OAP which reflects propgagtion direction and in-turn flips left-right (and it stores it like this too)

i.e. total effect is that data is flipped about origin.
________________________________________________________________________________________________

In [2]:
import dLux as dl
import dLux.utils as dlu

import jax.numpy as jnp
import numpy as np
import jax.random as jr
import jax.scipy as jsp
from jax import vmap  
import jax
jax.config.update("jax_enable_x64", True)
jax.config.update("jax_debug_nans", False)
jax.config.update('jax_disable_jit', False)


import zodiax as zdx
import optax
from tqdm.notebook import tqdm
# from tqdm import tqdm

from skimage.io import imread
from skimage.filters import window
import skimage as ski
from skimage.transform import resize

import matplotlib.pyplot as plt
from matplotlib.colors import PowerNorm, CenteredNorm
import matplotlib.colors as mcolors

from src.PhaseRetrieval import OptimManager, JointOptimManager, PointSource, TransmissiveLayer, DynamicAperture
from src.uv import dummy_splodge_mask, compute_complex_vis, UVComponents

import os
import pickle


import matplotlib as mpl
inferno = mpl.colormaps["inferno"]
viridis = mpl.colormaps["viridis"]
bwr = mpl.colormaps["bwr"]

inferno.set_bad("k", 0.5)
viridis.set_bad("k", 0.5)
bwr.set_bad("k", 0.5)
plt.rcParams['image.cmap'] = 'inferno'
plt.rcParams["font.family"] = "serif"
plt.rcParams["image.origin"] = 'upper' # true reading of array
plt.rcParams['figure.dpi'] = 72
plt.rcParams['figure.figsize'] = (10,10)
plt.rcParams["axes.titlesize"] = 18
plt.rcParams["figure.titlesize"] = 18
plt.rcParams["axes.labelsize"] = 15

data_dir = "/import/morgana1/snert/gpir9156/toliman/"


In [3]:
print(jax.lib.xla_bridge.get_backend().platform)

cpu


/tmp/ipykernel_807790/582351387.py:1: DeprecationWarning: jax.lib.xla_bridge.get_backend is deprecated; use jax.extend.backend.get_backend.
  print(jax.lib.xla_bridge.get_backend().platform)


In [ ]:
# ------- Physical Parameters ---------------------------------------------------------------------#
aperture_npix = 512           # Number of pixels across the aperture
aperture_diameter = 122e-3    # (m) slightly smaller for mask cap
spider_width = 20e-3          # Spider width (m)
spider_angle =270             # Spider angle (degrees), clockwise, 0 is spider pointing vertically up
coords = dlu.pixel_coords(npixels=aperture_npix, diameter=aperture_diameter)
circle = dlu.circle(coords=coords, radius=aperture_diameter/2) 

# Observations wavelengths (bandpass of 530-640nm)
red_laser_wl =  635e-09  # for laser data
green_laser_wl = 520e-09  # for laser data
wf_npixels = aperture_npix  # Number of pixels across the wavefront
wf_diam = aperture_diameter             # Diameter of initial wavefront to propagate wavefront (m)

# Detector parameters (BFS-U3-200S6-BD)
BFS_px_sep = 2.4e-6 *1e3        # pixel separation (mm)
f_det = 1338 # 1300#1350                    # Focal length from OAP2 to detector (mm) 
px_ang_sep = 2*np.arctan( (BFS_px_sep/2)/f_det ) # angular sep between pixels (rad)

# Simulated Detector
psf_npix = 40                 # Number of pixels along one dim of the PSF
psf_hlf_sz = int(psf_npix/2)             # half window sz of cropped data
oversample = 1                 # Oversampling factor for the PSF
psf_pixel_scale = dlu.rad2arcsec(px_ang_sep) # arcsec (to match detector plate scale) 80e-4 

# loading in psf pixel scale and det aberrations found in System_Aberrations.ipynb
f_aberr = "data/spider/retrieval_results/12_01_2026_mask/mcmc_system_coeffs_dict.pkl"

syst_noll = jnp.arange(4, 16) # only first 14 Zernike modes (excluding piston and tip/tilt) are used to classify syst aberrations
# syst_coeffs = pickle.load(open(f_aberr, "rb"))["value"]
stats_dict = pickle.load(open(f_aberr, "rb"))
syst_coeffs=[]
for noll_idx in syst_noll:
    syst_coeffs.append(stats_dict["Noll "+str(noll_idx)]["mean"])

syst_coeffs = jnp.array(syst_coeffs)    
syst_basis = dlu.zernike_basis(js=syst_noll, diameter=aperture_diameter, coordinates=coords)
print("System Coefficients (noll {}): {}".format(syst_noll,syst_coeffs))
psf_pixel_scale = stats_dict["Px Scale 0"]["mean"] # arcsec
print("Using detector pixel scale of {:.3f} arcsec".format(psf_pixel_scale))
